In [63]:
# =====================================================
# 04 Feature Engineering - Rider-Level Churn Dataset
# =====================================================

import pandas as pd
import numpy as np

# =====================================================
# Load Cleaned Datasets
# =====================================================

riders = pd.read_csv("../data/processed/riders_clean.csv")
drivers = pd.read_csv("../data/processed/drivers_clean.csv")
trips = pd.read_csv("../data/processed/trips_clean.csv")
sessions = pd.read_csv("../data/processed/sessions_clean.csv")

# =====================================================
# Convert Date Columns
# =====================================================

trips["pickup_time"] = pd.to_datetime(trips["pickup_time"], errors="coerce", utc=True)
trips["dropoff_time"] = pd.to_datetime(trips["dropoff_time"], errors="coerce", utc=True)
riders["signup_date"] = pd.to_datetime(riders["signup_date"], errors="coerce", utc=True)
drivers["signup_date"] = pd.to_datetime(drivers["signup_date"], errors="coerce", utc=True)
drivers["last_active"] = pd.to_datetime(drivers["last_active"], errors="coerce", utc=True)
sessions["session_time"] = pd.to_datetime(sessions["session_time"], errors="coerce", utc=True)

# =====================================================
# Snapshot Date
# =====================================================

snapshot_date = trips["pickup_time"].max()
print("Snapshot date:", snapshot_date)

# =====================================================
# Merge Trips with Driver Information
# =====================================================

trips_drivers = trips.merge(
    drivers[[
        "driver_id",
        "rating",
        "vehicle_type",
        "acceptance_rate"
    ]],
    on="driver_id",
    how="left"
)

# =====================================================
# Trip-Level Features Before Aggregation
# =====================================================

trips_drivers["fare_per_minute"] = (
    trips_drivers["fare"] / trips_drivers["trip_duration_minutes"]
).replace([np.inf, -np.inf], 0).fillna(0)

trips_drivers["tip_percentage"] = (
    trips_drivers["tip"] / trips_drivers["fare"] * 100
).replace([np.inf, -np.inf], 0).fillna(0)

trips_drivers["pickup_hour"] = trips_drivers["pickup_time"].dt.hour

trips_drivers["is_weekend"] = (
    trips_drivers["pickup_time"].dt.dayofweek >= 5
).astype(int)

trips_drivers["peak_hour_trip"] = (
    trips_drivers["pickup_hour"].between(7, 9) |
    trips_drivers["pickup_hour"].between(16, 19)
).astype(int)

trips_drivers["night_trip"] = (
    (trips_drivers["pickup_hour"] >= 22) |
    (trips_drivers["pickup_hour"] <= 5)
).astype(int)

trips_drivers["surge_trip"] = (
    trips_drivers["surge_multiplier"] > 1
).astype(int)

# =====================================================
# Aggregate Trip Features Per Rider
# =====================================================

trip_features = (
    trips_drivers
    .groupby("user_id")
    .agg(
        total_trips=("trip_id", "nunique"),
        avg_fare=("fare", "mean"),
        total_fare=("fare", "sum"),
        avg_surge=("surge_multiplier", "mean"),
        avg_tip=("tip", "mean"),
        avg_tip_percentage=("tip_percentage", "mean"),
        avg_trip_duration=("trip_duration_minutes", "mean"),
        avg_fare_per_minute=("fare_per_minute", "mean"),
        avg_driver_rating=("rating", "mean"),
        avg_driver_acceptance=("acceptance_rate", "mean"),
        weekend_trip_rate=("is_weekend", "mean"),
        peak_hour_trip_rate=("peak_hour_trip", "mean"),
        night_trip_rate=("night_trip", "mean"),
        surge_trip_rate=("surge_trip", "mean"),
        most_recent_trip=("pickup_time", "max"),
        first_trip=("pickup_time", "min"),
        favourite_payment_type=("payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
        favourite_weather=("weather", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
        favourite_vehicle_type=("vehicle_type", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
    )
    .reset_index()
)

# =====================================================
# Aggregate Session Features Per Rider
# =====================================================

session_features = (
    sessions
    .groupby("rider_id")
    .agg(
        total_sessions=("session_id", "nunique"),
        avg_time_on_app=("time_on_app", "mean"),
        avg_pages_visited=("pages_visited", "mean"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
    .rename(columns={"rider_id": "user_id"})
)

# =====================================================
# Merge Rider + Trip + Session Features
# =====================================================

model_df = riders.merge(
    trip_features,
    on="user_id",
    how="left"
)

model_df = model_df.merge(
    session_features,
    on="user_id",
    how="left"
)

# =====================================================
# Fill Missing Values
# =====================================================

numeric_fill_cols = [
    "total_trips",
    "avg_fare",
    "total_fare",
    "avg_surge",
    "avg_tip",
    "avg_tip_percentage",
    "avg_trip_duration",
    "avg_fare_per_minute",
    "avg_driver_rating",
    "avg_driver_acceptance",
    "weekend_trip_rate",
    "peak_hour_trip_rate",
    "night_trip_rate",
    "surge_trip_rate",
    "total_sessions",
    "avg_time_on_app",
    "avg_pages_visited",
    "conversion_rate"
]

model_df[numeric_fill_cols] = model_df[numeric_fill_cols].fillna(0)

categorical_fill_cols = [
    "favourite_payment_type",
    "favourite_weather",
    "favourite_vehicle_type"
]

model_df[categorical_fill_cols] = model_df[categorical_fill_cols].fillna("Unknown")

# =====================================================
# Customer Tenure and Recency Features
# =====================================================

model_df["account_tenure_days"] = (
    snapshot_date - model_df["signup_date"]
).dt.days

model_df["days_since_last_trip"] = (
    snapshot_date - model_df["most_recent_trip"]
).dt.days

model_df["days_since_first_trip"] = (
    snapshot_date - model_df["first_trip"]
).dt.days

model_df["days_since_last_trip"] = model_df["days_since_last_trip"].fillna(999)
model_df["days_since_first_trip"] = model_df["days_since_first_trip"].fillna(999)

model_df["account_tenure_days"] = (
    model_df["account_tenure_days"]
    .fillna(model_df["account_tenure_days"].median())
)

# =====================================================
# Behavioural Features
# =====================================================

model_df["trip_velocity"] = (
    model_df["total_trips"] /
    np.clip(model_df["account_tenure_days"], 1, None)
)

model_df["trips_per_session"] = (
    model_df["total_trips"] /
    np.clip(model_df["total_sessions"], 1, None)
)

model_df["engagement_score"] = (
    model_df["total_sessions"] * 0.4 +
    model_df["avg_time_on_app"] * 0.4 +
    model_df["avg_pages_visited"] * 0.2
)

model_df["is_referred"] = model_df["referred_by"].notna().astype(int)

# =====================================================
# Age Group
# =====================================================

model_df["age_group"] = pd.cut(
    model_df["age"],
    bins=[18, 25, 35, 45, 60, 100],
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
)

model_df["age_group"] = (
    model_df["age_group"]
    .cat.add_categories("Unknown")
    .fillna("Unknown")
)

# =====================================================
# Target Variable
# Churned if no trip in the last 30 days
# =====================================================

model_df["is_churned"] = (
    model_df["days_since_last_trip"] > 30
).astype(int)

print("Target counts:")
print(model_df["is_churned"].value_counts())

print("\nTarget percentage:")
print(model_df["is_churned"].value_counts(normalize=True) * 100)

# =====================================================
# Remove Leakage Columns
# =====================================================

model_df.drop(
    columns=[
        "churn_prob",
        "most_recent_trip",
        "first_trip",
        "days_since_last_trip",
        "days_since_first_trip"
    ],
    inplace=True,
    errors="ignore"
)

# =====================================================
# Save Rider-Level Modelling Dataset
# =====================================================

model_df.to_csv(
    "../data/processed/rider_level_churn_dataset.csv",
    index=False
)

print("Rider-level churn dataset saved successfully.")
print("Final shape:", model_df.shape)

Snapshot date: 2025-04-27 23:43:26+00:00
Target counts:
is_churned
0    8114
1    1886
Name: count, dtype: int64

Target percentage:
is_churned
0    81.14
1    18.86
Name: proportion, dtype: float64
Rider-level churn dataset saved successfully.
Final shape: (10000, 35)


In [64]:
# =====================================================
# Final Validation Checks
# =====================================================

print("Dataset shape:", model_df.shape)

print("\nDuplicate columns:")
print(model_df.columns[model_df.columns.duplicated()])

print("\nDuplicate User IDs:")
print(model_df["user_id"].duplicated().sum())

print("\nMissing values:")
missing = model_df.isnull().sum()
print(missing[missing > 0])

print("\nColumns:")
print(model_df.columns.tolist())

Dataset shape: (10000, 35)

Duplicate columns:
Index([], dtype='str')

Duplicate User IDs:
0

Missing values:
Series([], dtype: int64)

Columns:
['user_id', 'signup_date', 'loyalty_status', 'age', 'city', 'avg_rating_given', 'referred_by', 'is_referred', 'is_churned', 'total_trips', 'avg_fare', 'total_fare', 'avg_surge', 'avg_tip', 'avg_tip_percentage', 'avg_trip_duration', 'avg_fare_per_minute', 'avg_driver_rating', 'avg_driver_acceptance', 'weekend_trip_rate', 'peak_hour_trip_rate', 'night_trip_rate', 'surge_trip_rate', 'favourite_payment_type', 'favourite_weather', 'favourite_vehicle_type', 'total_sessions', 'avg_time_on_app', 'avg_pages_visited', 'conversion_rate', 'account_tenure_days', 'trip_velocity', 'trips_per_session', 'engagement_score', 'age_group']


In [65]:
# =====================================================
# Ensure One Row Per Rider
# =====================================================

assert model_df["user_id"].is_unique, "There are duplicate user_id values."

assert model_df.columns.duplicated().sum() == 0, "There are duplicate column names."

print("Validation passed: one row per rider and no duplicate columns.")

Validation passed: one row per rider and no duplicate columns.


differnce

In [1]:
import pandas as pd
import numpy as np

# Dictionary containing dataset names and date columns
data_files = {
    "sessions": ["session_time"],
    "trips": ["pickup_time", "dropoff_time"],
    "riders": ["signup_date"],
    "drivers": ["signup_date", "last_active"],
    "promotions": ["start_date", "end_date"]
}

# Dictionary to store DataFrames
dfs = {}

# Load all datasets
for name, date_cols in data_files.items():
    dfs[name] = pd.read_csv(
        f"../data/raw/{name}.csv",
        parse_dates=date_cols
    )

# Assign DataFrames to variables
sessions = dfs["sessions"]
trips = dfs["trips"]
riders = dfs["riders"]
drivers = dfs["drivers"]
promotions = dfs["promotions"]

In [2]:
# Load Cleaned Datasets
# ==================================================

riders = pd.read_csv("../data/processed/riders_clean.csv")

drivers = pd.read_csv("../data/processed/drivers_clean.csv")

trips = pd.read_csv("../data/processed/trips_clean.csv")

sessions = pd.read_csv("../data/processed/sessions_clean.csv")

In [4]:
# ==================================================
# Aggregate Session Features Per Rider
# ==================================================

sessions_features = (
    sessions
    .groupby("rider_id")
    .agg(
        total_sessions=("session_id", "count"),
        avg_time_on_app=("time_on_app", "mean"),
        avg_pages_visited=("pages_visited", "mean"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
)

# Rename for merging
sessions_features.rename(
    columns={"rider_id": "user_id"},
    inplace=True
)

In [5]:

display(sessions_features.head())

,user_id,total_sessions,avg_time_on_app,avg_pages_visited,conversion_rate
0,R00000,4,92.000000,3.000000,0.25
1,R00001,3,174.666667,2.666667,0.00
2,R00002,3,191.000000,3.000000,0.00
3,R00003,3,75.333333,1.666667,0.00
4,R00004,2,17.000000,2.500000,0.00


In [6]:
# ==================================================
# Merge Trips with Riders
# ==================================================

merged_df = trips.merge(
    riders,
    on="user_id",
    how="left"
)

print("Shape after merging Riders:", merged_df.shape)

Shape after merging Riders: (200000, 26)


In [7]:
# ==================================================
# Merge Drivers
# ==================================================

merged_df = merged_df.merge(
    drivers,
    on="driver_id",
    how="left",
    suffixes=("", "_driver")
)

print("Shape after merging Drivers:", merged_df.shape)

Shape after merging Drivers: (200000, 32)


In [8]:
# ==================================================
# Merge Session Features
# ==================================================

merged_df = merged_df.merge(
    sessions_features,
    on="user_id",
    how="left"
)

print("Shape after merging Sessions:", merged_df.shape)

Shape after merging Sessions: (200000, 36)


In [9]:
# ==================================================
# Validate Merged Dataset
# ==================================================

print("Rows:", merged_df.shape[0])
print("Columns:", merged_df.shape[1])

display(merged_df.head())

Rows: 200000
Columns: 36


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,...,rating,vehicle_type,signup_date_driver,last_active,city,acceptance_rate,total_sessions,avg_time_on_app,avg_pages_visited,conversion_rate
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50+00:00,2024-11-27 17:06:50+00:00,-1.108123,...,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628,4.0,21.25,1.00,0.00
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48+00:00,2024-10-28 23:12:48+00:00,6.675266,...,4.9,Suv,2023-05-06,2025-04-12 08:36:21.207528,Lagos,0.629250,5.0,51.80,2.40,0.20
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41+00:00,2025-02-17 03:25:41+00:00,-1.248589,...,4.7,Sedan,2023-08-09,2025-04-19 05:19:12.026949,Nairobi,0.990000,4.0,21.25,3.25,0.00
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14+00:00,2024-06-18 17:27:14+00:00,29.819554,...,3.9,Sedan,2023-04-22,2025-02-08 22:15:23.685705,Cairo,0.247121,2.0,80.50,1.00,0.50
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16+00:00,2024-10-05 08:01:16+00:00,-1.676479,...,4.2,Sedan,2024-08-22,2025-04-13 11:24:50.358526,Nairobi,0.611031,4.0,283.50,3.75,0.25


In [10]:
# ==================================================
# Missing Values
# ==================================================

missing = (
    merged_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

display(missing[missing > 0])

acceptance_rate       9048
city                  9048
last_active           9048
signup_date_driver    9048
vehicle_type          9048
rating                9048
total_sessions        1417
avg_time_on_app       1417
conversion_rate       1417
avg_pages_visited     1417
dtype: int64

In [11]:
# ==================================================
# Duplicate Trip IDs
# ==================================================

print(
    "Duplicate Trip IDs:",
    merged_df["trip_id"].duplicated().sum()
)

Duplicate Trip IDs: 0


In [12]:
# ==================================================
# Validate Row Count
# ==================================================

print("Trips:", len(trips))
print("Merged:", len(merged_df))

Trips: 200000
Merged: 200000


In [13]:
# ==================================================
# Replace Missing Session Features
# ==================================================

session_cols = [
    "total_sessions",
    "avg_time_on_app",
    "avg_pages_visited",
    "conversion_rate"
]

merged_df[session_cols] = merged_df[session_cols].fillna(0)

In [14]:
# Driver IDs in Trips but not in Drivers

missing_drivers = (
    trips.loc[
        ~trips["driver_id"].isin(drivers["driver_id"]),
        "driver_id"
    ]
    .unique()
)

print("Missing Driver IDs:", len(missing_drivers))

print(missing_drivers[:20])

Missing Driver IDs: 228
<ArrowStringArray>
['D01006', 'D04087', 'D03818', 'D00896', 'D02412', 'D04766', 'D01153',
 'D00893', 'D00451', 'D04701', 'D03884', 'D03057', 'D01337', 'D00148',
 'D03027', 'D04602', 'D01017', 'D03253', 'D01278', 'D04730']
Length: 20, dtype: str


In [15]:
# ==================================================
# Check Referential Integrity: Trips vs Drivers
# ==================================================

missing_drivers = (
    trips.loc[
        ~trips["driver_id"].isin(drivers["driver_id"]),
        "driver_id"
    ]
    .unique()
)

print("Missing Driver IDs:", len(missing_drivers))

missing_driver_trips = trips[
    trips["driver_id"].isin(missing_drivers)
]

print("Trips affected by missing drivers:", missing_driver_trips.shape[0])

Missing Driver IDs: 228
Trips affected by missing drivers: 9048


In [16]:
merged_df["rating"] = merged_df["rating"].fillna(
    merged_df["rating"].median()
)

merged_df["acceptance_rate"] = merged_df["acceptance_rate"].fillna(
    merged_df["acceptance_rate"].median()
)

In [17]:
merged_df["vehicle_type"] = merged_df["vehicle_type"].fillna("Unknown")

In [18]:
# ==================================================
# Driver Information Available
# ==================================================

merged_df["driver_info_missing"] = (
    merged_df["vehicle_type"] == "Unknown"
).astype(int)

In [19]:
# Fill session feature

session_cols = [
    "total_sessions",
    "avg_time_on_app",
    "avg_pages_visited",
    "conversion_rate"
]

merged_df[session_cols] = merged_df[session_cols].fillna(0)

In [20]:
print("Final Dataset Shape:", merged_df.shape)

print("\nMissing Values:")
print(merged_df.isnull().sum())

Final Dataset Shape: (200000, 37)

Missing Values:
trip_id                     0
user_id                     0
driver_id                   0
fare                        0
surge_multiplier            0
tip                         0
payment_type                0
pickup_time                 0
dropoff_time                0
pickup_lat                  0
pickup_lng                  0
dropoff_lat                 0
dropoff_lng                 0
weather                     0
city_x                      0
loyalty_status_x            0
trip_duration_minutes       0
signup_date                 0
loyalty_status_y            0
age                         0
city_y                      0
avg_rating_given            0
churn_prob                  0
referred_by                 0
is_referred                 0
is_churned                  0
rating                      0
vehicle_type                0
signup_date_driver       9048
last_active              9048
city                     9048
acceptance_rate    

In [21]:
# ==================================================
# Check Whether Duplicate Columns Match
# ==================================================

print(
    "City matches:",
    (merged_df["city_x"] == merged_df["city_y"]).all()
)

print(
    "Loyalty Status matches:",
    (merged_df["loyalty_status_x"] == merged_df["loyalty_status_y"]).all()
)

City matches: True
Loyalty Status matches: True


In [22]:
# ==================================================
# Remove Duplicate Columns
# ==================================================

merged_df.rename(
    columns={
        "city_x": "city",
        "loyalty_status_x": "loyalty_status"
    },
    inplace=True
)

merged_df.drop(
    columns=[
        "city_y",
        "loyalty_status_y"
    ],
    inplace=True
)

In [23]:
# ==================================================
# Save Final Merged Dataset
# ==================================================

merged_df.to_csv(
    "../data/processed/merged_dataset.csv",
    index=False
)

print("Merged dataset saved successfully.")

Merged dataset saved successfully.


In [24]:
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")

In [25]:
# ==================================================
# Convert Date Columns
# ==================================================

date_columns = [
    "pickup_time",
    "dropoff_time",
    "signup_date",
    "signup_date_driver",
    "last_active"
]

for col in date_columns:
    merged_df[col] = pd.to_datetime(
        merged_df[col],
        errors="coerce",
        utc=True
    )

In [26]:
# Time-Based Features

# ==================================================
# Pickup Hour
# ==================================================

merged_df["pickup_hour"] = (
    merged_df["pickup_time"].dt.hour
)


# ==================================================
# Day of Week
# ==================================================

merged_df["pickup_day"] = (
    merged_df["pickup_time"].dt.day_name()
)


# ==================================================
# Month
# ==================================================

merged_df["pickup_month"] = (
    merged_df["pickup_time"].dt.month_name()
)


# ==================================================
# Weekend Indicator
# ==================================================

merged_df["is_weekend"] = (
    merged_df["pickup_time"].dt.dayofweek >= 5
).astype(int)

In [27]:
# ==================================================
# Peak Hour Trips
# Morning:
# 7-9 AM
#
# Evening:
# 4-7 PM
# ==================================================

merged_df["peak_hour_trip"] = np.where(

    (
        merged_df["pickup_hour"].between(7,9)
    )
    |
    (
        merged_df["pickup_hour"].between(16,19)
    ),

    1,

    0
)

In [28]:
# ==================================================
# Night Trips
# Trips between 10 PM and 5 AM
# ==================================================

merged_df["night_trip"] = (
    (merged_df["pickup_hour"] >= 22) |
    (merged_df["pickup_hour"] <= 5)
).astype(int)

In [29]:
# ==================================================
# Driver Tenure
# ==================================================

merged_df["driver_tenure_days"] = (

    merged_df["pickup_time"]

    -

    merged_df["signup_date_driver"]

).dt.days

In [30]:
# ==================================================
# Days Since Driver Was Last Active
# ==================================================

merged_df["days_since_last_active"] = (

    merged_df["pickup_time"]

    -

    merged_df["last_active"]

).dt.days.abs()

In [31]:
# ==================================================
# Fare Per Minute
# ==================================================

merged_df["fare_per_minute"] = (
    merged_df["fare"] /
    merged_df["trip_duration_minutes"]
)

merged_df["fare_per_minute"] = (
    merged_df["fare_per_minute"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


# ==================================================
# Tip Percentage
# ==================================================

merged_df["tip_percentage"] = (
    merged_df["tip"] /
    merged_df["fare"]
) * 100

merged_df["tip_percentage"] = (
    merged_df["tip_percentage"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [32]:
# ==================================================
# Calculate Trip Distance (km)
# ==================================================

from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c


merged_df["trip_distance_km"] = haversine(
    merged_df["pickup_lat"],
    merged_df["pickup_lng"],
    merged_df["dropoff_lat"],
    merged_df["dropoff_lng"]
)

In [33]:
# ==================================================
# High Rated Driver
# ==================================================

merged_df["high_rated_driver"] = (
    merged_df["rating"] >= 4.5
).astype(int)


# ==================================================
# High Acceptance Driver
# ==================================================

merged_df["high_acceptance_driver"] = (
    merged_df["acceptance_rate"] >= 0.80
).astype(int)

In [34]:
# ==================================================
# Rider Age Groups
# ==================================================

merged_df["age_group"] = pd.cut(

    merged_df["age"],

    bins=[18,25,35,45,60,100],

    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-60",
        "60+"
    ]
)

In [35]:
# ==================================================
# Heavy App User
# ==================================================

merged_df["heavy_app_user"] = (
    merged_df["avg_time_on_app"] >=
    merged_df["avg_time_on_app"].median()
).astype(int)


# ==================================================
# Frequent User
# ==================================================

merged_df["frequent_user"] = (
    merged_df["total_sessions"] >= 5
).astype(int)

In [36]:
# ==================================================
# High Fare Trip
# ==================================================

merged_df["high_fare_trip"] = (
    merged_df["fare"] >=
    merged_df["fare"].quantile(0.75)
).astype(int)


# ==================================================
# Surge Applied
# ==================================================

merged_df["surge_trip"] = (
    merged_df["surge_multiplier"] > 1
).astype(int)

In [37]:
# ==================================================
# Rider Engagement Score
# ==================================================

merged_df["engagement_score"] = (

    merged_df["total_sessions"] * 0.4

    +

    merged_df["avg_time_on_app"] * 0.4

    +

    merged_df["avg_pages_visited"] * 0.2
)

In [38]:
# =====================================================
# Correct Churn Target Creation From Days Since Last Trip
# =====================================================

# Ensure pickup_time is datetime
merged_df["pickup_time"] = pd.to_datetime(
    merged_df["pickup_time"],
    errors="coerce",
    utc=True
)

# Create snapshot date using the latest trip date in the dataset
snapshot_date = merged_df["pickup_time"].max()

print("Snapshot date:", snapshot_date)

# Calculate the most recent trip per user
last_trip_per_user = (
    merged_df
    .groupby("user_id")["pickup_time"]
    .max()
    .reset_index()
    .rename(columns={"pickup_time": "most_recent_trip"})
)

# Merge most recent trip back to the full dataset
merged_df = merged_df.merge(
    last_trip_per_user,
    on="user_id",
    how="left"
)

# Calculate days since last trip
merged_df["days_since_last_trip"] = (
    snapshot_date - merged_df["most_recent_trip"]
).dt.days

# Users with no trip activity are treated as inactive
merged_df["days_since_last_trip"] = (
    merged_df["days_since_last_trip"]
    .fillna(999)
)

# Create churn target
# Business rule: churned if no trip in the last 30 days
merged_df["is_churned"] = (
    merged_df["days_since_last_trip"] > 30
).astype(int)

# Check new churn distribution
print("Target counts:")
print(merged_df["is_churned"].value_counts())

print("\nTarget percentage:")
print(merged_df["is_churned"].value_counts(normalize=True) * 100)

Snapshot date: 2025-04-27 23:43:26+00:00
Target counts:
is_churned
0    165193
1     34807
Name: count, dtype: int64

Target percentage:
is_churned
0    82.5965
1    17.4035
Name: proportion, dtype: float64


In [39]:
# ==================================================
# Save Engineered Dataset
# ==================================================

merged_df.to_csv(
    "../data/processed/featured_dataset.csv",
    index=False
)

In [41]:
# ==================================================
# Save Engineered Dataset
# ==================================================

merged_df.to_csv(
    "../data/processed/featured_dataset.csv",
    index=False
)

print("Feature engineering completed successfully.")

Feature engineering completed successfully.


In [43]:
# =====================================================
#  Column Cleanup
# =====================================================

# Rename duplicate trip_city column (driver city)
cols = list(merged_df.columns)

trip_city_positions = [
    i for i, col in enumerate(cols)
    if col == "trip_city"
]

if len(trip_city_positions) > 1:
    cols[trip_city_positions[1]] = "driver_city"

merged_df.columns = cols

# Verify
print("City columns:")
print([col for col in merged_df.columns if "city" in col.lower()])

print("\nDuplicate columns:")
print(merged_df.columns[merged_df.columns.duplicated()])

City columns:
['city', 'city']

Duplicate columns:
Index(['city'], dtype='str')


In [44]:
# Dataset shape
print(merged_df.shape)

# Missing values
print(merged_df.isnull().sum().sum())

# Duplicate rows
print(merged_df.duplicated().sum())

# Preview
display(merged_df.head())

(200000, 56)
54157
0


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,...,high_rated_driver,high_acceptance_driver,age_group,heavy_app_user,frequent_user,high_fare_trip,surge_trip,engagement_score,most_recent_trip,days_since_last_trip
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50+00:00,2024-11-27 17:06:50+00:00,-1.108123,...,0,0,36-45,0,0,0,0,10.30,2025-03-27 21:21:32+00:00,31
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48+00:00,2024-10-28 23:12:48+00:00,6.675266,...,1,0,36-45,0,1,0,0,23.20,2025-04-14 17:46:52+00:00,13
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41+00:00,2025-02-17 03:25:41+00:00,-1.248589,...,1,1,36-45,0,0,1,0,10.75,2025-04-15 16:03:01+00:00,12
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14+00:00,2024-06-18 17:27:14+00:00,29.819554,...,0,0,18-25,1,0,0,0,33.20,2025-04-26 01:02:15+00:00,1
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16+00:00,2024-10-05 08:01:16+00:00,-1.676479,...,0,0,26-35,1,0,0,0,115.75,2025-04-26 10:27:11+00:00,1


In [45]:
# ==================================================
# Dataset Information
# ==================================================

merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 56 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   trip_id                 200000 non-null  str                
 1   user_id                 200000 non-null  str                
 2   driver_id               200000 non-null  str                
 3   fare                    200000 non-null  float64            
 4   surge_multiplier        200000 non-null  float64            
 5   tip                     200000 non-null  float64            
 6   payment_type            200000 non-null  str                
 7   pickup_time             200000 non-null  datetime64[us, UTC]
 8   dropoff_time            200000 non-null  datetime64[us, UTC]
 9   pickup_lat              200000 non-null  float64            
 10  pickup_lng              200000 non-null  float64            
 11  dropoff_lat             200000 non-nu

In [46]:
# ==================================================
# Missing Values
# ==================================================

missing = merged_df.isnull().sum()

print(missing[missing > 0])

signup_date_driver        9048
last_active               9048
city                      9048
driver_tenure_days        9048
days_since_last_active    9048
age_group                 8917
dtype: int64


In [47]:
# ==================================================
# Duplicate Records
# ==================================================

print("Duplicate Rows:", merged_df.duplicated().sum())

Duplicate Rows: 0


In [49]:
# ==================================================
# Check for Duplicate Records
# ==================================================

duplicate_rows = merged_df.duplicated().sum()

print(f"Duplicate Rows: {duplicate_rows}")

if duplicate_rows == 0:
    print("No duplicate records found.")
else:
    print(f" {duplicate_rows} duplicate records detected.")

Duplicate Rows: 0
No duplicate records found.


In [50]:
# ==================================================
# Check for Missing Values
# ==================================================

missing_values = (
    merged_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage (%)": round(
        (missing_values / len(merged_df)) * 100,
        2
    )
})

# Display only columns with missing values
missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
]

print("=" * 60)
print("Missing Values Report")
print("=" * 60)

if missing_summary.empty:
    print(" No missing values found. Dataset is ready for modelling.")
else:
    display(missing_summary)

Missing Values Report


,Missing Values,Percentage (%)
driver_tenure_days,9048,4.52
city,9048,4.52
signup_date_driver,9048,4.52
days_since_last_active,9048,4.52
last_active,9048,4.52
age_group,8917,4.46


In [51]:
# ==================================================
# Handle Missing Driver Features
# ==================================================

merged_df["driver_tenure_days"] = (
    merged_df["driver_tenure_days"]
    .fillna(merged_df["driver_tenure_days"].median())
)

merged_df["days_since_last_active"] = (
    merged_df["days_since_last_active"]
    .fillna(merged_df["days_since_last_active"].median())
)

In [52]:
# ==================================================
# Fill Missing Age Groups
# ==================================================

merged_df["age_group"] = (
    merged_df["age_group"]
    .cat.add_categories("Unknown")
    .fillna("Unknown")
)

In [53]:
# ==================================================
# Fill Missing Driver City
# ==================================================

merged_df["city"] = merged_df["city"].fillna("Unknown")

In [54]:
# ==================================================
# Drop Raw Date Columns
# ==================================================

merged_df.drop(
    columns=[
        "signup_date_driver",
        "last_active"
    ],
    inplace=True
)

In [55]:
# ==================================================
# Final Missing Values Check
# ==================================================

missing = (
    merged_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing[missing > 0]
)

Series([], dtype: int64)

In [56]:
print(merged_df.columns.tolist())

['trip_id', 'user_id', 'driver_id', 'fare', 'surge_multiplier', 'tip', 'payment_type', 'pickup_time', 'dropoff_time', 'pickup_lat', 'pickup_lng', 'dropoff_lat', 'dropoff_lng', 'weather', 'city', 'loyalty_status', 'trip_duration_minutes', 'signup_date', 'age', 'avg_rating_given', 'churn_prob', 'referred_by', 'is_referred', 'is_churned', 'rating', 'vehicle_type', 'city', 'acceptance_rate', 'total_sessions', 'avg_time_on_app', 'avg_pages_visited', 'conversion_rate', 'driver_info_missing', 'pickup_hour', 'pickup_day', 'pickup_month', 'is_weekend', 'peak_hour_trip', 'night_trip', 'driver_tenure_days', 'days_since_last_active', 'fare_per_minute', 'tip_percentage', 'trip_distance_km', 'high_rated_driver', 'high_acceptance_driver', 'age_group', 'heavy_app_user', 'frequent_user', 'high_fare_trip', 'surge_trip', 'engagement_score', 'most_recent_trip', 'days_since_last_trip']


In [57]:
# =====================================================
# Rename the Second 'city' Column to 'driver_city'
# =====================================================

cols = list(merged_df.columns)

# Find the positions of all 'city' columns
city_positions = [i for i, col in enumerate(cols) if col == "city"]

print(city_positions)   # Should print something like [14, 26]

# Rename the second city column
cols[city_positions[1]] = "driver_city"

# Apply the updated column names
merged_df.columns = cols

# Check the result
print([col for col in merged_df.columns if "city" in col.lower()])

[14, 26]
['city', 'driver_city']


In [58]:
# =====================================================
# Rename Trip City
# =====================================================

merged_df.rename(
    columns={"city": "trip_city"},
    inplace=True
)

In [61]:
# =====================================================
# Remove Data Leakage Columns
# =====================================================

merged_df.drop(
    columns=[
        "churn_prob",
        "most_recent_trip",
        "days_since_last_trip"
    ],
    inplace=True,
    errors="ignore"
)

print("Remaining columns:", len(merged_df.columns))
print(merged_df.columns.tolist())

Remaining columns: 51
['trip_id', 'user_id', 'driver_id', 'fare', 'surge_multiplier', 'tip', 'payment_type', 'pickup_time', 'dropoff_time', 'pickup_lat', 'pickup_lng', 'dropoff_lat', 'dropoff_lng', 'weather', 'trip_city', 'loyalty_status', 'trip_duration_minutes', 'signup_date', 'age', 'avg_rating_given', 'referred_by', 'is_referred', 'is_churned', 'rating', 'vehicle_type', 'driver_city', 'acceptance_rate', 'total_sessions', 'avg_time_on_app', 'avg_pages_visited', 'conversion_rate', 'driver_info_missing', 'pickup_hour', 'pickup_day', 'pickup_month', 'is_weekend', 'peak_hour_trip', 'night_trip', 'driver_tenure_days', 'days_since_last_active', 'fare_per_minute', 'tip_percentage', 'trip_distance_km', 'high_rated_driver', 'high_acceptance_driver', 'age_group', 'heavy_app_user', 'frequent_user', 'high_fare_trip', 'surge_trip', 'engagement_score']


In [62]:
# =====================================================
# Save Feature Engineered Dataset
# =====================================================

merged_df.to_csv(
    "../data/processed/featured_dataset.csv",
    index=False
)

print("Feature engineering completed successfully.")

Feature engineering completed successfully.
